# Phase 5 — Recommendation Engine
## Fase 5 — Motor de Recomendación

**EN:** Compares the reader profile cluster centroids against the book catalog  
to find the most semantically similar titles. Each cluster gets its own top-10 recommendation list.

**ES:** Compara los centroides del perfil lector contra el catálogo de libros  
para encontrar los títulos más similares semánticamente. Cada cluster obtiene su propio top-10.


## Step 1 — Check GPU / Verificar GPU

**EN:** This phase vectorizes thousands of books. GPU acceleration reduces runtime from ~1 hour to ~5 minutes.  
If no GPU is detected, go to Runtime → Change runtime type → T4 GPU.

**ES:** Esta fase vectoriza miles de libros. La GPU reduce el tiempo de ~1 hora a ~5 minutos.  
Si no se detecta GPU, ve a Runtime → Change runtime type → T4 GPU.


In [ ]:
!pip install transformers -q
!pip install torch -q

import torch

if torch.cuda.is_available():
    device = torch.device('cuda')
    print(f"GPU available: {torch.cuda.get_device_name(0)}")
    print(f"GPU memory   : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
else:
    device = torch.device('cpu')
    print("WARNING: No GPU detected. Using CPU — vectorization will be slow.")
    print("Go to Runtime -> Change runtime type -> T4 GPU")

print(f"Active device: {device}")


Go to Runtime -> Change runtime type -> T4 GPU
Active device: cpu


## Step 2 — Imports and Mount Drive / Imports y Montar Drive


In [ ]:
import logging
import os
import pickle

import numpy as np
import pandas as pd
from sklearn.preprocessing import normalize
from sklearn.metrics.pairwise import cosine_similarity
from transformers import AutoTokenizer, AutoModel
import torch

from google.colab import drive

logging.basicConfig(
    level=logging.INFO,
    format="%(levelname)s:%(name)s:%(message)s"
)
logger = logging.getLogger("phase_5")

drive.mount('/content/drive')

DRIVE_PATH = "/content/drive/MyDrive/Colab Notebooks/PF_Ironhack/"

if torch.cuda.is_available():
    device = torch.device('cuda')
else:
    device = torch.device('cpu')

logger.info("Imports complete. Drive mounted. Device: %s", device)


Mounted at /content/drive


## Step 3 — Load Reader Profile and Catalog / Cargar Perfil Lector y Catálogo

**EN:** Loads the reader profile from Phase 4 and the clean book catalog from Phase 1.  
**ES:** Carga el perfil lector de la Fase 4 y el catálogo limpio de libros de la Fase 1.


In [ ]:
def load_pickle(filename: str, drive_path: str) -> object:
    """Loads a pickle file from Drive. Raises FileNotFoundError if not found."""
    filepath = os.path.join(drive_path, filename)
    if not os.path.exists(filepath):
        raise FileNotFoundError(
            f"{filename} not found at {filepath}. Run previous phase first."
        )
    with open(filepath, 'rb') as f:
        obj = pickle.load(f)
    logger.info("Loaded: %s", filename)
    return obj


def save_pickle(obj: object, filename: str, drive_path: str) -> None:
    """Serializes an object to a pickle file and logs the result."""
    filepath = os.path.join(drive_path, filename)
    with open(filepath, 'wb') as f:
        pickle.dump(obj, f)
    size_mb = os.path.getsize(filepath) / (1024 * 1024)
    logger.info("Saved: %s (%.1f MB)", filename, size_mb)
    print(f"  {filename:<40} {size_mb:.1f} MB")


reader_profile = load_pickle('reader_profile.pkl', DRIVE_PATH)

catalog_path = os.path.join(DRIVE_PATH, 'books_clean.csv')
df_catalog   = pd.read_csv(catalog_path)

notes_path = os.path.join(DRIVE_PATH, 'user_notes_clustered.csv')
df_notes   = pd.read_csv(notes_path)

print(f"Catalog books   : {len(df_catalog):,}")
print(f"Books read      : {len(df_notes)}")
print(f"Profile clusters: {len(reader_profile)}")
print()
print("Reader profile clusters:")
for key, cluster in reader_profile.items():
    label = cluster.get('label', f"Cluster {cluster['cluster_id']}")
    print(f"  Cluster {cluster['cluster_id']} — {label}")
    print(f"    Books: {', '.join(cluster['books'][:4])}{'...' if len(cluster['books']) > 4 else ''}")


Catalog books   : 78,811
Books read      : 50
Profile clusters: 5

Reader profile clusters:
  Cluster 0 — Inner Worlds
    Books: Ask the Dust, Crime and Punishment, Factotum, Wilt...
  Cluster 1 — Speculative Minds
    Books: Dune, Sapiens, Ubik, A Supposedly Fun Thing I'll Never Do Again...
  Cluster 2 — Political & Dystopian
    Books: 1984, The Catcher in the Rye, Frankenstein, Brave New World...
  Cluster 3 — The Seeker's Path
    Books: The Name of the Wind, The Little Prince, Pride and Prejudice, The Royal Game...
  Cluster 4 — The Dark & The Strange
    Books: Let the Right One In, The Metamorphosis, The Raven, The Fall of the House of Usher...


## Step 4 — Catalog Embeddings: Load or Generate
## Embeddings del Catálogo: Cargar o Generar

**EN:**
- If `catalog_embeddings.pkl` already exists in Drive → load it directly.  
- If it does not exist → generate it and save it immediately.

**ES:**  
- Si `catalog_embeddings.pkl` ya existe en Drive → lo cargamos directamente.  
- Si no existe → lo generamos y guardamos inmediatamente.


In [ ]:
catalog_embeddings_path = os.path.join(DRIVE_PATH, 'catalog_embeddings.pkl')

if os.path.exists(catalog_embeddings_path):
    # Catalog embeddings already generated — load directly, skip BERT
    catalog_embeddings = load_pickle('catalog_embeddings.pkl', DRIVE_PATH)
    logger.info("Catalog embeddings loaded from Drive. Shape: %s", str(catalog_embeddings.shape))
    print(f"Catalog embeddings loaded: {catalog_embeddings.shape}")
    print("Skipping BERT vectorization — run Steps 5 and 6 only on first run.")

else:
    print("catalog_embeddings.pkl not found. Running BERT vectorization...")
    print("Continue to Steps 5 and 6.")

Catalog embeddings loaded: (78811, 768)
Skipping BERT vectorization — run Steps 5 and 6 only on first run.


## Step 5 — Load BERT Model / Cargar Modelo BERT

**EN:** Skip this step if `catalog_embeddings.pkl` was already loaded in Step 4.  
**ES:** Omite este paso si `catalog_embeddings.pkl` ya se cargó en el Step 4.


In [ ]:
if not os.path.exists(catalog_embeddings_path):
    MODEL_NAME = "sentence-transformers/all-mpnet-base-v2"

    logger.info("Loading BERT model: %s", MODEL_NAME)
    tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
    model     = AutoModel.from_pretrained(MODEL_NAME).to(device)
    model.eval()

    logger.info(
        "Model loaded on %s. Parameters: %s",
        device,
        f"{sum(p.numel() for p in model.parameters()):,}"
    )
    print(f"Model ready: {MODEL_NAME} on {device}")
else:
    print("Catalog embeddings already loaded — BERT model not needed.")

config.json:   0%|          | 0.00/571 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/363 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/438M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

MPNetModel LOAD REPORT from: sentence-transformers/all-mpnet-base-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Model ready: sentence-transformers/all-mpnet-base-v2 on cuda


## Step 6 — Vectorize Catalog / Vectorizar Catálogo

**EN:** Skip this step if `catalog_embeddings.pkl` was already loaded in Step 4.  
**ES:** Omite este paso si `catalog_embeddings.pkl` ya se cargó en el Step 4.

In [ ]:
def get_embeddings_batch(
    texts: list, tokenizer, model, device, batch_size: int = 64
) -> np.ndarray:
    """
    Vectorizes a list of texts in batches using GPU acceleration.

    Args:
        texts:      List of strings to vectorize
        tokenizer:  Loaded BERT tokenizer
        model:      BERT model on GPU
        device:     torch.device ('cuda' or 'cpu')
        batch_size: Number of texts per batch (64 is optimal for T4 GPU)

    Returns:
        numpy array of shape (len(texts), 768)
    """
    all_embeddings = []
    total = len(texts)

    for start in range(0, total, batch_size):
        end   = min(start + batch_size, total)
        batch = texts[start:end]

        inputs = tokenizer(
            batch,
            padding=True,
            truncation=True,
            max_length=512,       # all-mpnet-base-v2 soporta hasta 514 tokens
            return_tensors='pt'
        )
        # Mover inputs al mismo dispositivo que el modelo
        inputs = {k: v.to(device) for k, v in inputs.items()}

        with torch.no_grad():
            outputs = model(**inputs)

        token_emb = outputs.last_hidden_state        # (batch, n_tokens, 768)
        mask      = inputs['attention_mask']
        mask_exp  = mask.unsqueeze(-1).expand(token_emb.size()).float()
        batch_emb = (token_emb * mask_exp).sum(1) / mask_exp.sum(1).clamp(min=1e-9)

        # Mover de GPU a CPU antes de convertir a numpy
        all_embeddings.append(batch_emb.cpu().numpy())

        if (start // batch_size) % 8 == 0:
            print(f"  Processed: {end:,} / {total:,} ({end / total * 100:.0f}%)")

    return np.vstack(all_embeddings)  # (total, 768)


if not os.path.exists(catalog_embeddings_path):
    logger.info("Vectorizing %d catalog books...", len(df_catalog))
    texts = df_catalog['description'].tolist()

    catalog_embeddings_raw = get_embeddings_batch(
        texts, tokenizer, model, device, batch_size=64
    )
    # Normalización L2: los vectores tendrán longitud = 1
    # La similitud coseno refleja entonces dirección (significado), no magnitud
    catalog_embeddings = normalize(catalog_embeddings_raw, norm='l2')

    logger.info("Vectorización completa. Shape: %s", str(catalog_embeddings.shape))
    print()
    print(f"Shape de embeddings: {catalog_embeddings.shape[0]:,} libros x {catalog_embeddings.shape[1]} dimensiones")
    print("Normalización L2   : aplicada")

    # Save immediately — this file never needs to be regenerated
    print()
    print("Guardando en Drive...")
    save_pickle(catalog_embeddings, 'catalog_embeddings.pkl', DRIVE_PATH)

else:
    print("Skipped — catalog embeddings already saved.")

  Processed: 64 / 78,811 (0%)
  Processed: 576 / 78,811 (1%)
  Processed: 1,088 / 78,811 (1%)
  Processed: 1,600 / 78,811 (2%)
  Processed: 2,112 / 78,811 (3%)
  Processed: 2,624 / 78,811 (3%)
  Processed: 3,136 / 78,811 (4%)
  Processed: 3,648 / 78,811 (5%)
  Processed: 4,160 / 78,811 (5%)
  Processed: 4,672 / 78,811 (6%)
  Processed: 5,184 / 78,811 (7%)
  Processed: 5,696 / 78,811 (7%)
  Processed: 6,208 / 78,811 (8%)
  Processed: 6,720 / 78,811 (9%)
  Processed: 7,232 / 78,811 (9%)
  Processed: 7,744 / 78,811 (10%)
  Processed: 8,256 / 78,811 (10%)
  Processed: 8,768 / 78,811 (11%)
  Processed: 9,280 / 78,811 (12%)
  Processed: 9,792 / 78,811 (12%)
  Processed: 10,304 / 78,811 (13%)
  Processed: 10,816 / 78,811 (14%)
  Processed: 11,328 / 78,811 (14%)
  Processed: 11,840 / 78,811 (15%)
  Processed: 12,352 / 78,811 (16%)
  Processed: 12,864 / 78,811 (16%)
  Processed: 13,376 / 78,811 (17%)
  Processed: 13,888 / 78,811 (18%)
  Processed: 14,400 / 78,811 (18%)
  Processed: 14,912 / 78,

## Step 7 — Compute Recommendations / Calcular Recomendaciones

**EN:** For each cluster centroid, we compute cosine similarity against all catalog embeddings.

Two filters are applied before computing recommendations:
- **Already read** — exact match + known title aliases (e.g. "Nineteen Eighty-Four" = "1984")
- **Academic titles** — pattern matching on title to exclude literary criticism and essays.

**ES:** Para cada centroide de cluster, calculamos similitud coseno contra todos los embeddings.

Se aplican dos filtros antes de calcular las recomendaciones:
- **Ya leídos** — coincidencia exacta + aliases conocidos (ej. "Nineteen Eighty-Four" = "1984")
- **Títulos académicos** — patrones en el título para excluir crítica literaria y ensayos.

In [ ]:
import re
import ast

TOP_N = 10

# df_notes es el nombre usado en Cell 6 al cargar user_notes_clustered.csv
titles_read = df_notes['title'].str.lower().str.strip().tolist()

# -------------------------------------------------------
# Filtro 1 — Libros ya leídos (con aliases de títulos conocidos)
# -------------------------------------------------------
# Algunos libros aparecen con títulos alternativos en el catálogo
# (ej: "Nineteen Eighty-Four" = "1984") — los aliases evitan que
# se recomienden libros que ya están en las notas con otro nombre.
TITLE_ALIASES = {
    "nineteen eighty-four": "1984",
    "the trial": "el proceso",
    # añadir más si aparecen
}

def is_already_read(catalog_title: str) -> bool:
    """Devuelve True si el título del catálogo coincide con algún libro ya leído."""
    catalog_lower = str(catalog_title).lower().strip()
    # Comprobar alias conocidos
    canonical = TITLE_ALIASES.get(catalog_lower, catalog_lower)
    return any(read_title in canonical or read_title in catalog_lower
               for read_title in titles_read)

# -------------------------------------------------------
# Filtro 2 — Títulos académicos
# -------------------------------------------------------
# Captura crítica literaria y ensayos
TITLE_EXCLUDE_PATTERNS = [
    r'\bessays\b',
    r'\btales\b',
    r'\bselected stories\b',
    r'\the complete stories\b',
    r'\bmatters\b',
    r'\bthe best of\b',
    r'\ba life\b',
    r'\brevealed\b',
    r'\bbiography\b',
    r'\blife of\b',
    r'\bletters\b',
    r'\bjournal\b',
    r'\bcriticism\b',
    r'\bcompanion\b',
    r'\bguide to\b',
    r'\bfield guide\b',
    r'\bjudging a book\b',
    r'\bwhat makes this book\b',
    r'\bwhy i write\b',
    r'\bthe art of\b',
    r'\bhow literature\b',
    r'\byou\'ve got to read\b',
    r'\byear of reading\b',
    r'\bhow fiction\b',
]

def has_excluded_pattern(title: str) -> bool:
    """Devuelve True si el título indica crítica literaria o ensayo."""
    title_lower = str(title).lower()
    return any(re.search(pattern, title_lower) for pattern in TITLE_EXCLUDE_PATTERNS)

# Construir máscaras y combinarlas
mask_not_read    = ~df_catalog['title'].apply(is_already_read)
mask_not_academic = ~df_catalog['title'].apply(has_excluded_pattern)
mask_final        = mask_not_read & mask_not_academic

not_read_indices = set(df_catalog[mask_final].index.tolist())

print(f"Total catalog books     : {len(df_catalog):,}")
print(f"Already read (excluded) : {(~mask_not_read).sum()}")
print(f"Excluded (academic)     : {(~mask_not_academic & mask_not_read).sum():,}")
print(f"Candidate books         : {len(not_read_indices):,}")
print()

# -------------------------------------------------------
# Calcular recomendaciones por cluster
# -------------------------------------------------------

recommendations  = {}
recommended_globally = set()  # títulos ya recomendados en cualquier cluster

for key, cluster in reader_profile.items():
    c        = cluster['cluster_id']
    centroid = cluster['centroid']
    label    = cluster.get('label', f'Cluster {c}')

    sims = cosine_similarity(centroid.reshape(1, -1), catalog_embeddings)[0]

    top_indices = [
        i for i in sims.argsort()[::-1]
        if i in not_read_indices
    ][:TOP_N * 5]  # margen amplio para sobrevivir ambos filtros de deduplicación

    # Deduplicación dentro del cluster Y entre clusters
    seen_this_cluster = set()
    top_indices_deduped = []

    for i in top_indices:
        title_norm = re.sub(r'[^a-z0-9]', '', str(df_catalog.iloc[i]['title']).lower())

        # Excluir si ya aparece en este cluster o en cualquier cluster anterior
        if title_norm in seen_this_cluster or title_norm in recommended_globally:
            continue

        seen_this_cluster.add(title_norm)
        top_indices_deduped.append(i)

        if len(top_indices_deduped) == TOP_N:
            break

    # Registrar globalmente los títulos recomendados en este cluster
    recommended_globally.update(seen_this_cluster)

    cluster_recs = []
    for rank, idx in enumerate(top_indices_deduped):
        row = df_catalog.iloc[idx]
        cluster_recs.append({
            'rank':             rank + 1,
            'cluster_id':       c,
            'cluster_label':    label,
            'cluster_books':    ', '.join(cluster['books']),
            'title':            row['title'],
            'author_names':     str(row['author_names']).strip("[]'\""),
            'similarity_score': round(float(sims[idx]), 4),
            'description':      str(row['description'])[:200],
        })

    recommendations[f'cluster_{c}'] = cluster_recs

    print(f"Cluster {c} — {label}")
    print(f"  Basado en: {', '.join(cluster['books'][:3])}...")
    print(f"  Keywords : {', '.join(cluster['top_words'][:4])}")
    print(f"  Top {TOP_N} recomendaciones:")
    for rec in cluster_recs:
        print(f"    {rec['rank']}. {rec['title']} — {rec['author_names']} (score: {rec['similarity_score']})")
    print()

logger.info("Recomendaciones calculadas para %d clusters.", len(recommendations))


Total catalog books     : 78,811
Already read (excluded) : 407
Excluded (academic)   : 1,728
Candidate books         : 76,676

Cluster 0 — Inner Worlds
  Basado en: Ask the Dust, Crime and Punishment, Factotum...
  Keywords : identity, kind, self, honesty
  Top 10 recomendaciones:
    1. Reader’s Block — David Markson (score: 0.7316)
    2. This Is Not a Novel — David Markson (score: 0.7314)
    3. Zeno's Conscience — Italo Svevo, William Weaver (score: 0.7292)
    4. Identity — Milan Kundera, Linda Asher (score: 0.72)
    5. The Night in Question — Tobias Wolff (score: 0.7167)
    6. The Curtain: An Essay in Seven Parts — Milan Kundera, Linda Asher (score: 0.7145)
    7. G. — John Berger (score: 0.7044)
    8. Everyman — Philip Roth (score: 0.6958)
    9. All That Man Is — David Szalay (score: 0.6947)
    10. The Insufferable Gaucho — Roberto Bolano, Chris    Andrews (score: 0.694)

Cluster 1 — Speculative Minds
  Basado en: Dune, Sapiens, Ubik...
  Keywords : history, reality, scienc

## Step 8 — Save Recommendations / Guardar Recomendaciones

**EN:** Saves recommendations in two formats:  
- `.pkl` — used by RAG engine and Streamlit app  
- `.csv` — human-readable for verification

**ES:** Guarda las recomendaciones en dos formatos:  
- `.pkl` — usado por el motor RAG y la app Streamlit  
- `.csv` — legible por humanos para verificación


In [ ]:
save_pickle(recommendations, 'recommendations.pkl', DRIVE_PATH)

# Convertir a DataFrame para guardar también en CSV
all_recs = []
for cluster_recs in recommendations.values():
    all_recs.extend(cluster_recs)

df_recs   = pd.DataFrame(all_recs)
recs_path = os.path.join(DRIVE_PATH, 'recommendations.csv')
df_recs.to_csv(recs_path, index=False)
print(f"  {'recommendations.csv':<40} guardado ({len(df_recs)} recomendaciones)")

print()
print("Verificación — archivos en Drive:")
print("-" * 55)
for fname in ['catalog_embeddings.pkl', 'recommendations.pkl', 'recommendations.csv']:
    path   = os.path.join(DRIVE_PATH, fname)
    status = "OK     " if os.path.exists(path) else "MISSING"
    print(f"  [{status}] {fname}")


  recommendations.pkl                      0.0 MB
  recommendations.csv                      guardado (50 recomendaciones)

Verificación — archivos en Drive:
-------------------------------------------------------
  [OK     ] catalog_embeddings.pkl
  [OK     ] recommendations.pkl
  [OK     ] recommendations.csv


## Step 9 — Recommendation Quality Analysis / Análisis de Calidad

**EN:** Review whether recommended books align thematically with each cluster.  
A score > 0.5 indicates strong alignment. Scores < 0.3 suggest the cluster centroid  
may need more notes to become more representative.

**ES:** Revisamos si los libros recomendados se alinean temáticamente con cada cluster.  
Un score > 0.5 indica alta alineación. Scores < 0.3 sugieren que el centroide necesita  
más notas para ser más representativo.


In [ ]:
print("=" * 65)
print("ANÁLISIS DE CALIDAD DE RECOMENDACIONES")
print("=" * 65)
print()

for key, cluster_recs in recommendations.items():
    c            = cluster_recs[0]['cluster_id']
    cluster_info = reader_profile[f'cluster_{c}']
    label        = cluster_info.get('label', f'Cluster {c}')

    print(f"CLUSTER {c} — {label}")
    print(f"  Tus libros   : {', '.join(cluster_info['books'][:4])}{'...' if len(cluster_info['books']) > 4 else ''}")
    print(f"  Keywords     : {', '.join(cluster_info['top_words'][:4])}")
    print()
    print(f"  Top {len(cluster_recs)} recomendaciones:")
    print(f"  {'#':<3} {'Título':<40} {'Author'} {'Score':<8}")
    print(f"  {'-'*3} {'-'*40} {'-'*8} {'-'*25}")

    for rec in cluster_recs:
        title_short = rec['title'][:38] + '..' if len(rec['title']) > 38 else rec['title']
        author_short = rec['author_names'][:13] +  '..' if len(rec['author_names']) > 13 else rec['author_names']
        print(f"  {rec['rank']:<3} {title_short:<40} {author_short:<15} {rec['similarity_score']:<8}")

    avg_score = sum(r['similarity_score'] for r in cluster_recs) / len(cluster_recs)
    print()
    print(f"  Score medio (top {len(cluster_recs)}): {avg_score:.4f}")
    print()

print("-" * 65)
print("Referencia de scores: > 0.5 = alta alineación | < 0.3 = alineación débil")


ANÁLISIS DE CALIDAD DE RECOMENDACIONES

CLUSTER 0 — Inner Worlds
  Tus libros   : Ask the Dust, Crime and Punishment, Factotum, Wilt...
  Keywords     : identity, kind, self, honesty

  Top 10 recomendaciones:
  #   Título                                   Author Score   
  --- ---------------------------------------- -------- -------------------------
  1   Reader’s Block                           David Markson   0.7316  
  2   This Is Not a Novel                      David Markson   0.7314  
  3   Zeno's Conscience                        Italo Svevo, .. 0.7292  
  4   Identity                                 Milan Kundera.. 0.72    
  5   The Night in Question                    Tobias Wolff    0.7167  
  6   The Curtain: An Essay in Seven Parts     Milan Kundera.. 0.7145  
  7   G.                                       John Berger     0.7044  
  8   Everyman                                 Philip Roth     0.6958  
  9   All That Man Is                          David Szalay    0.6947